# Robot path planning by Q-learning

The path planning problem of Units 2 and 3, solved without a model.

A robot crosses a $13\times14$ grid from the left edge to the right edge,
moving one column to the right at every stage and choosing to go up,
straight, or down. Free cells cost $1$ to traverse straight and $\sqrt2$
diagonally; obstacle cells (`0` in the map) cost an extra $c=4$. The top
and bottom rows are walls, which push the robot back and also cost $c$.

The motion is **noisy**: commanding "up" moves the robot up with
probability $1-2p$ and leaves it straight with probability $2p$;
commanding "straight" leaves it straight with probability $1-2p$ and
slips one cell up or down with probability $p$ each.

Because the column index only ever increases, the last column is
absorbing, and the problem is a stochastic shortest path problem -- the
form `q_learning_ssp` expects. The state is the pair (row, column),
flattened to one index.

The agent is given no map and no slip probabilities. It is given a
simulator.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (14, 4)

from q_learning import q_learning_ssp

rng = np.random.default_rng(0)

In [ ]:
M = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
n1, n2 = M.shape
c_obst, p_slip = 4, 0.2

def ind2grid(x):
    return x % n1, x // n1

def grid2ind(i, k):
    return i + k * n1

print(f"{n1} rows x {n2} columns = {n1 * n2} states")

In [ ]:
def model_and_cost(x, u):
    """Simulator: 0 = up, 1 = straight, 2 = down. Column always advances."""
    i, k = ind2grid(x)
    if i == 0:                                 # top wall
        i_next, cost = 1, np.sqrt(2) + c_obst
    elif i == n1 - 1:                          # bottom wall
        i_next, cost = n1 - 2, np.sqrt(2) + c_obst
    elif M[i, k] == 0:                         # obstacle: no choice, pay extra
        i_next, cost = i, c_obst + 1
    else:
        r = rng.random()
        if u == 0:                             # commanded up
            i_next, cost = (i + 1, np.sqrt(2)) if r > 2 * p_slip else (i, 1.0)
        elif u == 1:                           # commanded straight
            if r < p_slip:         i_next, cost = i + 1, np.sqrt(2)
            elif r < 1 - p_slip:   i_next, cost = i,     1.0
            else:                  i_next, cost = i - 1, np.sqrt(2)
        else:                                  # commanded down
            i_next, cost = (i - 1, np.sqrt(2)) if r > 2 * p_slip else (i, 1.0)
    return grid2ind(i_next, k + 1), cost


# walls and obstacles offer no choice; free cells offer three actions
num_actions = np.array([1 if (M[ind2grid(x)] == 0 or ind2grid(x)[0] in (0, n1 - 1)) else 3
                        for x in range(n1 * n2)])
terminal = list(range(n1 * (n2 - 1), n1 * n2))

In [ ]:
Q, mu, J = q_learning_ssp(
    num_actions, model_and_cost,
    nsim=30_000, eps=0.5, alpha=0.2,
    prob_init_state=np.r_[np.ones(n1) / n1, np.zeros(n1 * (n2 - 1))],
    terminal_states=terminal, rng=rng)

Jplot = np.array([[J[grid2ind(i, k)] for k in range(n2)] for i in range(n1)])
# the plotting helper below expects actions numbered 1 = straight, 2 = up, 3 = down
uplot = [[{0: 2, 1: 1, 2: 3}[mu[grid2ind(i, k)]] if num_actions[grid2ind(i, k)] == 3 else 1
          for k in range(n2)] for i in range(n1)]
print("learned cost-to-go from each cell of the first column:")
print(np.round(Jplot[:, 0], 2))

In [ ]:
import matplotlib.patches as patches

# plotting helpers, taken unchanged from A_2_RobotPlanningStochastic.ipynb
def plotsquare(l, r, ax, color):
    ax.add_patch(patches.Rectangle((l, r), 1, 1, facecolor=color, edgecolor="k"))

def plottriangle(p, theta, ax):
    d1, d2 = 0.30, 0.06
    p2 = p + np.array([-d1 * np.cos(theta) - d2 * np.sin(theta),
                       d1 * np.sin(theta) - d2 * np.cos(theta)])
    p3 = p + np.array([-d1 * np.cos(theta) + d2 * np.sin(theta),
                       d1 * np.sin(theta) + d2 * np.cos(theta)])
    ax.add_patch(patches.Polygon(np.vstack([p3, p2, p]), closed=True,
                                 facecolor="k", edgecolor="k"))

def plotpathplanning(J, M, u):
    p, m = M.shape
    fig, ax = plt.subplots(1, figsize=(16, 9))
    ax.set_xlim(1, m + 1); ax.set_ylim(1, p + 1)
    for r in range(p):
        for l in range(m):
            plotsquare(l + 1, r + 1, ax, "grey" if M[r, l] == 0 else "white")
            ax.text(l + 1.2, r + 1.4, "%.2f" % J[r, l], fontsize=9)
    fig, ax = plt.subplots(1, figsize=(16, 9))
    ax.set_xlim(1, m + 1); ax.set_ylim(1, p + 1)
    for r in range(p):
        for l in range(m - 1):
            plotsquare(l + 1, r + 1, ax, "grey" if M[r, l] == 0 else "white")
            if M[r, l] == 0 or r in (0, p - 1):
                dr = 1 if r == 0 else (-1 if r == p - 1 else 0)
            else:
                dr = {1: 0, 2: 1, 3: -1}[u[r][l]]
            y0, y1 = r + 1.5 - 0.25 * dr, r + 1.5 + 0.25 * dr
            ax.plot([l + 1.25, l + 1.75], [y0, y1], "k")
            plottriangle(np.array([l + 1.75, y1]), -np.arctan2(dr, 1), ax)
    plt.show()


plotpathplanning(Jplot, M, uplot)

In [ ]:
def exact_stochastic_dp():
    """The same problem solved with the model, for comparison."""
    s2 = np.sqrt(2)
    J = np.zeros((n1, n2))
    for k in range(n2 - 2, -1, -1):
        for i in range(n1):
            if i == 0:         J[i, k] = s2 + c_obst + J[1, k + 1]
            elif i == n1 - 1:  J[i, k] = s2 + c_obst + J[n1 - 2, k + 1]
            elif M[i, k] == 0: J[i, k] = c_obst + 1 + J[i, k + 1]
            else:
                up    = (1 - 2*p_slip) * (s2 + J[i+1, k+1]) + 2*p_slip * (1 + J[i, k+1])
                strt  = p_slip * (s2 + J[i+1, k+1]) + (1 - 2*p_slip) * (1 + J[i, k+1]) \
                        + p_slip * (s2 + J[i-1, k+1])
                down  = (1 - 2*p_slip) * (s2 + J[i-1, k+1]) + 2*p_slip * (1 + J[i, k+1])
                J[i, k] = min(up, strt, down)
    return J


J_exact = exact_stochastic_dp()
print("first column -- exact :", np.round(J_exact[:, 0], 2))
print("first column -- learned:", np.round(Jplot[:, 0], 2))
print("max error over the whole grid: %.2f" % np.max(np.abs(J_exact - Jplot)[:, :-1]))
print("\nbest starting row -- exact %d, learned %d"
      % (1 + np.argmin(J_exact[1:n1-1, 0]), 1 + np.argmin(Jplot[1:n1-1, 0])))

Thirty thousand simulated crossings put the learned cost-to-go within a
few tenths of the exact stochastic dynamic programming values, and both
agree on which row is the best place to start.

The errors are not uniform. They are smallest in the first column, where
every simulation begins, and in the cells along the corridors the policy
actually uses; they are largest in cells the robot rarely enters, which
is the visitation condition of the convergence theorem showing up again,
more gently than in the inventory notebook.

It is worth comparing the effort. The exact solver sweeps the grid once,
backwards, and is done -- but it needs `M`, the costs, and the slip
probabilities. Q-learning needs none of those and pays for it with
thirty thousand trajectories.